# 🌿 Plant Disease Detector

Trains a CNN on the PlantVillage dataset (15 pepper / potato / tomato classes), then scores it on
**50 held-out test images** that are removed from the dataset *before* training, so the model has never seen them.

**Before you start:** Runtime → Change runtime type → **GPU** (training is ~10× faster than on CPU).

Pipeline: Kaggle setup → download → set aside 50 test samples → train/val split → train → save → test on the 50 samples.

## 1. Kaggle setup
Upload your `kaggle.json` (Kaggle → Settings → API → Create New Token). The upload result is not printed, so your key never ends up in the notebook.

In [ ]:
import os, shutil
from google.colab import files

KAGGLE_PATH = '/root/.kaggle/kaggle.json'
if not os.path.exists(KAGGLE_PATH):
    uploaded = files.upload()          # assigned, not displayed -> key is not printed
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.move(next(iter(uploaded)), KAGGLE_PATH)
    os.chmod(KAGGLE_PATH, 0o600)
print('Kaggle key installed.')

## 2. Download the dataset

In [ ]:
%cd /content
if not os.path.isdir('/content/PlantVillage'):
    !kaggle datasets download -d emmarex/plantdisease
    !unzip -oq plantdisease.zip
    os.remove('plantdisease.zip')
    # The zip ships a second copy of the same images under plantvillage/PlantVillage — drop it.
    if os.path.isdir('/content/plantvillage') and not os.path.samefile('/content/plantvillage', '/content/PlantVillage'):
        shutil.rmtree('/content/plantvillage')
print('Classes:', sorted(os.listdir('/content/PlantVillage')))

## 3. Set aside 50 test samples
Picks 50 images (3–4 per class, fixed seed so it's the same 50 every run) and **moves** them out of the
training folder into `/content/test_samples/<class>/`. Because they're moved, they can't leak into training or validation.

In [ ]:
import json, random, pathlib, shutil

DATA_DIR = pathlib.Path('/content/PlantVillage')
TEST_DIR = pathlib.Path('/content/test_samples')
N_TEST   = 50
IMG_EXTS = {'.jpg', '.jpeg', '.png'}
manifest_path = TEST_DIR / 'manifest.json'

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    # If the dataset was re-downloaded, make sure the test images aren't back in it.
    for item in manifest:
        (DATA_DIR / item['file']).unlink(missing_ok=True)
    print(f'Using the {len(manifest)} test samples already set aside.')
else:
    rng = random.Random(42)
    classes = sorted(d.name for d in DATA_DIR.iterdir() if d.is_dir())
    per_class = {c: N_TEST // len(classes) for c in classes}
    for c in rng.sample(classes, N_TEST % len(classes)):
        per_class[c] += 1

    manifest = []
    for c in classes:
        imgs = sorted(p for p in (DATA_DIR / c).iterdir() if p.suffix.lower() in IMG_EXTS)
        for p in rng.sample(imgs, per_class[c]):
            dest = TEST_DIR / c / p.name
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(p), dest)
            manifest.append({'file': f'{c}/{p.name}', 'label': c})
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print(f'Moved {len(manifest)} test samples to {TEST_DIR}')

from collections import Counter
for c, n in sorted(Counter(m['label'] for m in manifest).items()):
    print(f'  {n}  {c}')

## 4. Train / validation split
Same `validation_split` and seed for both subsets → 80% train, 20% validation, no overlap, nothing wasted.

In [ ]:
import tensorflow as tf
print('GPU:', tf.config.list_physical_devices('GPU') or 'none (training will be slow)')

IMG_SIZE   = (128, 128)
BATCH_SIZE = 32
SEED       = 123

train_data, val_data = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset='both',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)
class_names = train_data.class_names
num_classes = len(class_names)
print(num_classes, 'classes:', class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_data = train_data.prefetch(AUTOTUNE)
val_data   = val_data.prefetch(AUTOTUNE)

## 5. Model
Same CNN as before, plus:
- `Rescaling` **inside** the model, so the saved model takes raw images and test code can't forget to normalise;
- light data augmentation and `Dropout` to reduce the overfitting seen before (train 93% vs val 87%).

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),
    layers.Rescaling(1./255),
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax'),
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

## 6. Train
Stops early once validation loss stops improving and keeps the best weights.

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=[early_stop],
)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'], label='train')
ax1.plot(history.history['val_accuracy'], label='validation')
ax1.set_title('Accuracy'); ax1.set_xlabel('epoch'); ax1.legend()
ax2.plot(history.history['loss'], label='train')
ax2.plot(history.history['val_loss'], label='validation')
ax2.set_title('Loss'); ax2.set_xlabel('epoch'); ax2.legend()
plt.show()

## 7. Save the model
Saves the model and its class order so `batch_test.py` can reload them in a later session.

In [ ]:
MODEL_PATH       = '/content/plant_disease_model.keras'
CLASS_NAMES_PATH = '/content/class_names.json'

model.save(MODEL_PATH)
with open(CLASS_NAMES_PATH, 'w') as f:
    json.dump(class_names, f, indent=2)
print('Saved', MODEL_PATH, 'and', CLASS_NAMES_PATH)

## 8. Test on the 50 held-out samples
The test folders are named exactly like the training classes, so labels line up with `class_names`.

In [ ]:
import numpy as np

test_data = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    class_names=class_names,   # same label order as training
    shuffle=False,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

test_loss, test_acc = model.evaluate(test_data, verbose=0)
print(f'Test accuracy on {len(test_data.file_paths)} held-out images: {test_acc:.1%}  (loss {test_loss:.3f})')

probs  = model.predict(test_data, verbose=0)
y_pred = probs.argmax(axis=1)
y_true = np.concatenate([y.numpy() for _, y in test_data])

results = []
for path, t, p, pr in zip(test_data.file_paths, y_true, y_pred, probs):
    results.append({
        'file':       os.path.relpath(path, TEST_DIR),
        'true_label': class_names[t],
        'pred_label': class_names[p],
        'confidence': round(float(pr[p]) * 100, 2),
        'correct':    bool(t == p),
    })

print(f"\n{'Class':<45} Correct")
for i, c in enumerate(class_names):
    mask = y_true == i
    print(f'{c:<45} {int((y_pred[mask] == i).sum())}/{int(mask.sum())}')

wrong = [r for r in results if not r['correct']]
print(f'\nMisclassified ({len(wrong)}):')
for r in wrong:
    print(f"  {r['file']}\n    true: {r['true_label']}   predicted: {r['pred_label']} ({r['confidence']:.1f}%)")

report = {
    'total': len(results),
    'correct': len(results) - len(wrong),
    'accuracy_pct': round(test_acc * 100, 2),
    'per_image': results,
}
with open(TEST_DIR / 'test_report.json', 'w') as f:
    json.dump(report, f, indent=2)

In [ ]:
# Visual check: green = correct, red = wrong
cols = 10
rows = int(np.ceil(len(results) / cols))
plt.figure(figsize=(cols * 2, rows * 2.4))
for i, r in enumerate(results):
    img = tf.keras.utils.load_img(TEST_DIR / r['file'], target_size=IMG_SIZE)
    ax = plt.subplot(rows, cols, i + 1)
    ax.imshow(img); ax.axis('off')
    ax.set_title(f"{r['pred_label'][:18]}\n{r['confidence']:.0f}%", fontsize=7,
                 color='green' if r['correct'] else 'red')
plt.suptitle(f'Held-out test set — accuracy {test_acc:.1%}')
plt.tight_layout()
plt.show()

## 9. Download the 50 test samples and the report

In [ ]:
shutil.make_archive('/content/test_samples', 'zip', TEST_DIR)
files.download('/content/test_samples.zip')